# TEST-05: Flip Sequence (Entry/Exit Trade Counting)

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Precisely verify trade-counting semantics on known transitions.

**Signal sequence:** `[0, 0, +1, +1, +1, 0, 0, -1, -1, 0, +1, 0, ...]`

**Independent derivation** (signal at bar t → fill at bar t+1):

| Fill bar | Action | Event |
|---:|---|---|
| 2 | desired=0 → no change | flat |
| 3 | desired=+1 → entry long | round-trip 1 starts |
| 4 | desired=+1 → no change | |
| 5 | desired=+1 → no change | |
| 6 | desired=0 → exit long | round-trip 1 complete |
| 7 | desired=0 → no change | flat |
| 8 | desired=-1 → entry short | round-trip 2 starts |
| 9 | desired=-1 → no change | |
| 10 | desired=0 → exit short | round-trip 2 complete |
| 11 | desired=+1 → entry long | round-trip 3 starts |
| 12 | desired=0 → exit long | round-trip 3 complete |

**Expected:** 3 round-trip trades

In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

from lib.backtest import run_backtest

OHLCV_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet'
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10_000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

ohlcv = load_ohlcv(OHLCV_PATH)
n = len(ohlcv)
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))

flip_vals = [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, -1.0, -1.0, 0.0, 1.0, 0.0]
full_vals = flip_vals + [0.0] * (n - len(flip_vals))
sig = pd.Series(full_vals, index=ohlcv.index, name='signal')

result = run_backtest(sig, ohlcv, funding, **COSTS, execution='next_bar_open',
                      stream='klines', interval='4h', gaps_path=GAPS_PATH)
print(f'Signal sequence (first 12): {list(sig.iloc[:12].values)}')
print(f'Trade count: {len(result.trades_df)} (expected: 3)')

Signal sequence (first 12): [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, -1.0, -1.0, 0.0, 1.0, 0.0]
Trade count: 3 (expected: 3)


In [2]:
print('=== Trades detail ===')
for i, (_, row) in enumerate(result.trades_df.iterrows()):
    print(f'Round-trip {i+1}:')
    print(f'  Direction: {row["direction"]} ({"long" if row["direction"]==1 else "short"})')
    print(f'  Entry bar: {row["entry_bar"]}')
    print(f'  Exit bar:  {row["exit_bar"]}')
    print(f'  Net PnL:   {row["net_pnl"]:.2f}')
    print()

=== Trades detail ===
Round-trip 1:
  Direction: 1 (long)
  Entry bar: 2020-01-01 15:59:59.999000+00:00
  Exit bar:  2020-01-02 03:59:59.999000+00:00
  Net PnL:   -30.01

Round-trip 2:
  Direction: -1 (short)
  Entry bar: 2020-01-02 11:59:59.999000+00:00
  Exit bar:  2020-01-02 19:59:59.999000+00:00
  Net PnL:   -11.30

Round-trip 3:
  Direction: 1 (long)
  Entry bar: 2020-01-02 23:59:59.999000+00:00
  Exit bar:  2020-01-03 03:59:59.999000+00:00
  Net PnL:   -19.66



In [3]:
# Verify bar indices
entry_bar_1 = result.trades_df.iloc[0]['entry_bar']
exit_bar_1 = result.trades_df.iloc[0]['exit_bar']
entry_bar_2 = result.trades_df.iloc[1]['entry_bar']
exit_bar_2 = result.trades_df.iloc[1]['exit_bar']
entry_bar_3 = result.trades_df.iloc[2]['entry_bar']
exit_bar_3 = result.trades_df.iloc[2]['exit_bar']

print(f'RT1 entry = bar[3]: {entry_bar_1 == ohlcv.index[3]} (signal[2]=+1 -> fill at bar[3])')
print(f'RT1 exit  = bar[6]: {exit_bar_1 == ohlcv.index[6]}  (signal[5]=0 -> close at bar[6])')
print(f'RT2 entry = bar[8]: {entry_bar_2 == ohlcv.index[8]} (signal[7]=-1 -> fill at bar[8])')
print(f'RT2 exit  = bar[10]: {exit_bar_2 == ohlcv.index[10]} (signal[9]=0 -> close at bar[10])')
print(f'RT3 entry = bar[11]: {entry_bar_3 == ohlcv.index[11]} (signal[10]=+1 -> fill at bar[11])')
print(f'RT3 exit  = bar[12]: {exit_bar_3 == ohlcv.index[12]} (signal[11]=0 -> close at bar[12])')

all_ok = all([
    len(result.trades_df) == 3,
    result.trades_df.iloc[0]['direction'] == 1,
    result.trades_df.iloc[1]['direction'] == -1,
    result.trades_df.iloc[2]['direction'] == 1,
    entry_bar_1 == ohlcv.index[3],
    exit_bar_1 == ohlcv.index[6],
    entry_bar_2 == ohlcv.index[8],
    exit_bar_2 == ohlcv.index[10],
    entry_bar_3 == ohlcv.index[11],
    exit_bar_3 == ohlcv.index[12],
])
print(f'\n=== VERDICT: {"PASS" if all_ok else "FAIL"} ===')
print('Trade count, directions, and bar indices all match the independently derived expectation.')

RT1 entry = bar[3]: True (signal[2]=+1 -> fill at bar[3])
RT1 exit  = bar[6]: True  (signal[5]=0 -> close at bar[6])
RT2 entry = bar[8]: True (signal[7]=-1 -> fill at bar[8])
RT2 exit  = bar[10]: True (signal[9]=0 -> close at bar[10])
RT3 entry = bar[11]: True (signal[10]=+1 -> fill at bar[11])
RT3 exit  = bar[12]: True (signal[11]=0 -> close at bar[12])

=== VERDICT: PASS ===
Trade count, directions, and bar indices all match the independently derived expectation.
